# HELM — Inference & Model Internals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimchev/HELM/blob/master/notebooks/demo_inference.ipynb)

Explore what HELM receives as input, how the hierarchy is structured, what the model outputs, and how predictions map to leaf and ancestor labels.

## 1. Setup

In [ ]:
%cd /content
!rm -rf HELM
!git clone https://github.com/marjanstoimchev/HELM.git
%cd HELM
!pip install -q -r requirements.txt

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html 2>/dev/null || echo "PyG extensions installed via fallback"
print("Setup complete!")

In [ ]:
import os, sys, gc, glob, yaml
import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning import seed_everything
import matplotlib.pyplot as plt

sys.path.insert(0, ".")

from data.dataset_pipeline import DatasetPipeline
from data.hierarchy import create_edge_index, extract_paths, process_hierarchy_config, extend_ys, build_hierarchy_mapping
from datamodules.base_datamodule import BaseDataModule
from models.model import h_deit_base_embedding
from augmentations import Preprocess
from callbacks import ModelCheckpoint_, EarlyStopping_, RichProgressBar_
from utils.utils import Dotdict, calculate_metrics, predict
from trainers import get_trainer_class

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load hierarchy and dataset

In [ ]:
DATASET = "ucm"
DATASET_CONFIG = f"configs/dataset/{DATASET}.yaml"
SEED = 42

with open(DATASET_CONFIG) as f:
    ds_cfg = yaml.safe_load(f)

# Parse hierarchy
h_out = process_hierarchy_config(DATASET_CONFIG)
leaf_labels = ds_cfg["leaf_labels"]
leaf_paths = h_out.leaf_paths.to_dict()
intermediate_paths = h_out.intermediate_paths.to_dict()

print(f"Dataset: {ds_cfg['name']}")
print(f"Leaf labels (M_l): {h_out.num_classes}")
print(f"Intermediate nodes (M_h): {len(intermediate_paths)}")
print(f"Total nodes (M = M_l + M_h): {h_out.num_classes_extended}")

## 3. Hierarchy structure

The label hierarchy is a tree. Leaf nodes are the original dataset labels. Intermediate nodes are parent categories added by HELM.

In [ ]:
def print_tree(node, prefix="", is_last=True):
    """Print a YAML hierarchy dict as a tree."""
    keys = list(node.keys())
    for i, key in enumerate(keys):
        last = (i == len(keys) - 1)
        connector = "\u2514\u2500\u2500 " if last else "\u251c\u2500\u2500 "
        is_leaf = not node[key]  # empty dict = leaf
        suffix = " (leaf)" if is_leaf else ""
        print(f"{prefix}{connector}{key}{suffix}")
        if node[key]:
            extension = "    " if last else "\u2502   "
            print_tree(node[key], prefix + extension, last)

print(f"Hierarchy for {DATASET.upper()}:")
print_tree(ds_cfg["hierarchy"])

## 4. Leaf vs ancestor labels

In HMLC mode, when a leaf label is active, all its ancestors are activated too. This shows the mapping.

In [ ]:
# Build ordered leaf paths + intermediate paths
ordered_lp = {k: leaf_paths[k] for k in leaf_labels}
final_strings = {**ordered_lp, **intermediate_paths}
all_label_names = list(final_strings.keys())

# Reverse lookup: path code -> label name
code_to_name = {v: k for k, v in final_strings.items()}

# Show ancestor chain for each leaf
print(f"{'Leaf Label':<25s} {'Ancestors (root \u2192 leaf)'}")
print("\u2500" * 70)
hierarchy_map = build_hierarchy_mapping(final_strings)
for leaf in leaf_labels[:5]:  # show first 5
    ancestors_codes = hierarchy_map[leaf]
    chain = [code_to_name.get(c, c) for c in ancestors_codes if c in code_to_name]
    print(f"{leaf:<25s} {' \u2192 '.join(chain)}")
print(f"... ({len(leaf_labels)} total leaves)")

## 5. What the model sees

HELM receives images as 224x224 tensors. In HMLC mode, the label vector is extended from M_l leaf labels to M total labels (leaf + ancestors).

In [ ]:
from lightning import seed_everything
seed_everything(SEED, workers=True)

pipeline = DatasetPipeline(yaml_path=DATASET_CONFIG, seed=SEED, cache_dir="./Datasets/mlc_datasets_npy")
outputs = pipeline.run_pipeline(fraction_labeled=None)

num_leaves = pipeline.num_classes
num_classes = pipeline.num_classes_extended

print("\u2500\u2500 Model Input \u2500\u2500")
print(f"Image shape:         {outputs['X_te'][0].shape}  (C, H, W)")
print(f"Image dtype:         {outputs['X_te'][0].dtype}")
print(f"Image value range:   [{outputs['X_te'][0].min()}, {outputs['X_te'][0].max()}]")

print(f"\n\u2500\u2500 Label Vectors \u2500\u2500")
print(f"Leaf labels (Y_te):  shape {outputs['Y_te'].shape}  ({num_leaves} classes)")
print(f"Full labels (Y_te_h): shape {outputs['Y_te_h'].shape}  ({num_classes} classes = {num_leaves} leaf + {num_classes - num_leaves} ancestors)")

# Show an example
idx = 0
leaf_vec = outputs['Y_te'][idx]
full_vec = outputs['Y_te_h'][idx]
active_leaves = [leaf_labels[i] for i in np.where(leaf_vec > 0)[0]]
active_all = [all_label_names[i] for i in np.where(full_vec > 0)[0]]
active_ancestors = [n for n in active_all if n not in active_leaves]

print(f"\n\u2500\u2500 Example (sample {idx}) \u2500\u2500")
print(f"Active leaf labels:     {active_leaves}")
print(f"Activated ancestors:    {active_ancestors}")
print(f"Full extended vector:   {active_all}")

## 6. Visualize input samples with hierarchical labels

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "axes.titleweight": "bold",
})

N_SHOW = 4
indices = np.random.choice(len(outputs['X_te']), N_SHOW, replace=False)

fig, axes = plt.subplots(2, N_SHOW, figsize=(4.5 * N_SHOW, 8),
                          gridspec_kw={"height_ratios": [1, 1.3]})

for col, idx in enumerate(indices):
    # Image
    img = outputs['X_te'][idx]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    img = np.moveaxis(img, 0, -1)
    axes[0, col].imshow(np.clip(img, 0, 1))
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    # Labels
    leaf_vec = outputs['Y_te'][idx]
    full_vec = outputs['Y_te_h'][idx]
    active_leaves = [leaf_labels[i] for i in np.where(leaf_vec > 0)[0]]
    active_all = [all_label_names[i] for i in np.where(full_vec > 0)[0]]
    active_ancestors = [n for n in active_all if n not in active_leaves]

    axes[0, col].set_title(", ".join(active_leaves) or "none", fontsize=9)

    # Bar chart: full extended label vector
    ax = axes[1, col]
    active_idx = np.where(full_vec > 0)[0]
    names = [all_label_names[i] for i in active_idx]
    colors = ["#2E86AB" if all_label_names[i] in active_leaves else "#E8871E" for i in active_idx]

    if len(names) > 0:
        ax.barh(range(len(names)), [1]*len(names), color=colors, edgecolor="white")
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.5)
        ax.set_xticks([])
    if col == 0:
        ax.set_ylabel("Active labels", fontsize=10)

# Legend
from matplotlib.patches import Patch
fig.legend(
    [Patch(color="#2E86AB"), Patch(color="#E8871E")],
    ["Leaf label", "Ancestor (activated by hierarchy)"],
    loc="lower center", ncol=2, fontsize=10, frameon=False,
)
fig.suptitle(f"UCM \u2014 Images with Hierarchical Labels", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## 7. Load pre-trained HELM model

Downloads the pre-trained HELM checkpoint from Google Drive (UCM, `hmlc-sl-graph-byol`, 100% labeled) and runs inference.

In [ ]:
from models.model import h_deit_base_embedding
from trainers import get_trainer_class

METHOD = "hmlc-sl-graph-byol"
GDRIVE_FILE_ID = "106kboTGXisVJ6RWHmpGeBfkWyTE3vHqr"

with open(f"configs/method/{METHOD}.yaml") as f:
    mcfg = yaml.safe_load(f)

edge_index = create_edge_index(hierarchy=pipeline.label_to_predecessors)

config = Dotdict({
    "training": {"lr": 1e-4, "head_lr": 1e-4, "max_lr": 3e-4,
                 "apply_scheduler": False, "epochs": 1, "min_epochs": 1,
                 "patience": 5, "lr_schedule_patience": 5,
                 "accumulate_grad_batches": 1, "deterministic": True, "log_every_n_steps": 1},
    "dataset": {"name": DATASET, "folder_name": DATASET, "num_classes": num_leaves},
})

backbone = h_deit_base_embedding(num_classes=num_classes, pretrained=False)
trainer_cls = get_trainer_class(mcfg["training_mode"], mcfg["learning_task"], mcfg["apply_edges"], mcfg["apply_byol"])

# Download pre-trained checkpoint from Google Drive
model_dir = f"saved_models/{DATASET}/{METHOD}/fraction_100/seed_{SEED}"
model_path = os.path.join(model_dir, "model.ckpt")

if not os.path.exists(model_path):
    !pip install -q gdown
    import gdown
    os.makedirs(model_dir, exist_ok=True)
    print("Downloading pre-trained HELM checkpoint...")
    gdown.download(id=GDRIVE_FILE_ID, output=model_path, quiet=False)

# Load Lightning checkpoint
model = trainer_cls.load_from_checkpoint(
    model_path, config=config, backbone=backbone,
    num_leaves=num_leaves, learning_task=mcfg["learning_task"], edge_index=edge_index,
)
model.eval()
print(f"Loaded pre-trained HELM checkpoint from {model_path}")

# Show what the model outputs for a single sample
from augmentations import Preprocess
preprocess = Preprocess()
sample_img = outputs['X_te'][0]
x = preprocess(np.moveaxis(sample_img, 0, -1)).unsqueeze(0)
y_dummy = torch.zeros(1, num_classes)

with torch.no_grad():
    out = model._forward_eval(x, y_dummy) if hasattr(model, '_forward_eval') else model.forward(x, y_dummy)

logits = out['logits']
probs = torch.sigmoid(logits).squeeze().numpy()

print(f"\n── Model Output ──")
print(f"Logits shape: {logits.shape}  ({num_leaves} leaf classes)")
print(f"\nPredicted probabilities:")
for i, name in enumerate(leaf_labels):
    bar = "█" * int(probs[i] * 30)
    marker = " ◄" if probs[i] > 0.5 else ""
    print(f"  {name:<20s} {probs[i]:.3f} {bar}{marker}")

## 8. Prediction visualization

Top-5 predictions per test sample. Blue = correct leaf label, gray = incorrect.

In [ ]:
from datamodules.base_datamodule import BaseDataModule

transforms = Preprocess()
datamodule = BaseDataModule(outputs, batch_size=16, num_workers=2, transforms=transforms)

inference_trainer = L.Trainer(accelerator="auto", devices=[0], strategy="auto", enable_model_summary=False)
Y = predict(inference_trainer, model, datamodule)

N_SHOW = 4
n_test = Y["y_true"].shape[0]
indices = np.random.choice(n_test, N_SHOW, replace=False)

fig, axes = plt.subplots(2, N_SHOW, figsize=(4 * N_SHOW, 7),
                          gridspec_kw={"height_ratios": [1, 1.2]})

for col, idx in enumerate(indices):
    img = outputs["X_te"][idx]
    if img.max() > 1:
        img = img.astype(np.float32) / 255.0
    img = np.moveaxis(img, 0, -1)
    axes[0, col].imshow(np.clip(img, 0, 1))
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])

    true_idx = np.where(Y["y_true"][idx] > 0)[0]
    true_names = [leaf_labels[i] for i in true_idx if i < len(leaf_labels)]
    axes[0, col].set_title(", ".join(true_names) or "none", fontsize=9, wrap=True)

    ax = axes[1, col]
    scores = Y["y_scores"][idx][:len(leaf_labels)]
    top_k = np.argsort(scores)[-5:][::-1]
    top_names = [leaf_labels[k] for k in top_k]
    top_vals = [scores[k] for k in top_k]
    bar_colors = ["#2E86AB" if Y["y_true"][idx][k] > 0 else "#ccc" for k in top_k]

    ax.barh(range(len(top_k)), top_vals, color=bar_colors)
    ax.set_yticks(range(len(top_k)))
    ax.set_yticklabels(top_names, fontsize=9)
    ax.set_xlim(0, 1)
    ax.invert_yaxis()
    if col == 0:
        ax.set_ylabel("Top-5 predictions", fontsize=10)

plt.suptitle("HELM predictions \u2014 Blue = correct, Gray = incorrect", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()